In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict
import ast

import numpy as np
import pandas as pd
from pymongo import MongoClient
from pydantic import BaseModel
import yaml

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

LONG_THRESHOLD = 30.0


In [5]:
class EventsConfig(BaseModel):
    db: str
    url: str
    collections: Dict[str, str]
    year: str


def load_config(path: str | Path = "config/config.yaml") -> EventsConfig:
    path = Path(path)
    if not path.exists():
        path = Path("../config/config.yaml")

    with path.open("r", encoding="utf-8") as f:
        raw = yaml.safe_load(f)

    return EventsConfig(
        db=raw["mongo"]["db"],
        url=raw["mongo"]["url"],
        collection=raw["mongo"]["collection"],
        year=raw["season"]["year"],
    )


events_config = load_config()
events_config


EventsConfig(db='WhoScored', url='mongodb://localhost:27017/', collections={'collection_teams': 'available_teams', 'collection_schedule': 'game_schedule', 'collection_logs': 'error_logs', 'collection_raw_events': 'game_raw_events', 'collection_processed_events': 'game_processed_events', 'collection_team_game_stats': 'game_team_stats', 'collection_player_game_stats': 'game_player_stats'}, year='2009-2010')

In [ ]:
# client = MongoClient(events_config.url)
# db = client[events_config.db]
# db[events_config.collections.get("collection_player_game_stats")].delete_many({})

DeleteResult({'n': 8384, 'ok': 1.0}, acknowledged=True)

## Qualifier Helpers


In [2]:
# Dev notebooks use the same processing implementation as prod.
import sys

ROOT = Path.cwd()
if not (ROOT / "prod_pipeline").exists():
    ROOT = ROOT.parent
PROD_PIPELINE = ROOT / "prod_pipeline"
if str(PROD_PIPELINE) not in sys.path:
    sys.path.insert(0, str(PROD_PIPELINE))

from processed_events import ProcessedEvents
from opta_qualifiers import load_opta_qualifier_catalog, qualifier_catalog_entry


def build_processed_game_events(raw_events: pd.DataFrame) -> pd.DataFrame:
    return ProcessedEvents.build_processed_game_events(raw_events)


def ensure_qual_cols(df: pd.DataFrame) -> pd.DataFrame:
    return ProcessedEvents._ensure_qual_cols(df)


def has(df: pd.DataFrame, qual: str) -> pd.Series:
    return ProcessedEvents._has(df, qual)


def has_any(df: pd.DataFrame, quals) -> pd.Series:
    return ProcessedEvents._has_any(df, set(quals))


In [3]:
# build_processed_game_events is imported from prod_pipeline.processed_events above.
# Keeping this cell intentionally small prevents dev/prod processing logic from drifting.


In [10]:
from pymongo.errors import BulkWriteError

raw_events_collection = events_config.collections["collection_raw_events"]
processed_events_collection = events_config.collections["collection_processed_events"]


def normalize_mongo_value(value):
    if value is None:
        return None
    if isinstance(value, dict):
        return {k: normalize_mongo_value(v) for k, v in value.items()}
    if isinstance(value, list):
        return [normalize_mongo_value(v) for v in value]
    if isinstance(value, tuple):
        return [normalize_mongo_value(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value


def records_for_mongo(df: pd.DataFrame) -> list[dict]:
    return [
        {k: normalize_mongo_value(v) for k, v in record.items()}
        for record in df.to_dict(orient="records")
    ]


def season_pending_game_ids() -> list[int]:
    client = MongoClient(events_config.url)
    db = client[events_config.db]
    raw_collection = db[raw_events_collection]
    processed_collection = db[processed_events_collection]

    raw_ids = {
        int(game_id)
        for game_id in raw_collection.distinct("game_id", {"season": events_config.year})
        if game_id is not None
    }
    processed_ids = {
        int(game_id)
        for game_id in processed_collection.distinct("game_id", {"season": events_config.year})
        if game_id is not None
    }
    client.close()
    return sorted(raw_ids - processed_ids)


pending_game_ids = season_pending_game_ids()
print(f"Season: {events_config.year}")
print(f"Pending season games: {len(pending_game_ids):,}")
print(pending_game_ids[:10])


Season: 2025-2026
Pending season games: 279
[1908319, 1910598, 1910599, 1910600, 1910601, 1910602, 1910603, 1910604, 1910605, 1910606]


## Run Season Save


In [ ]:
LIMIT = 1

client = MongoClient(events_config.url)
db = client[events_config.db]
raw_collection = db[raw_events_collection]
processed_collection = db[processed_events_collection]
game_ids_to_process = pending_game_ids if LIMIT is None else pending_game_ids[:LIMIT]
inserted_games = 0
inserted_rows = 0

for game_id in game_ids_to_process[:1]:
    raw_events = pd.DataFrame(
        raw_collection.find({"season": events_config.year, "game_id": game_id}, {"_id": 0})
    )
    if raw_events.empty:
        print(f"game_id={game_id}: no raw rows")
        continue

    processed_events = build_processed_game_events(raw_events)
    records = records_for_mongo(processed_events)
    if not records:
        print(f"game_id={game_id}: no processed rows")
        continue

    processed_collection.delete_many({"season": events_config.year, "game_id": game_id})
    try:
        result = processed_collection.insert_many(records, ordered=False)
    except BulkWriteError:
        processed_collection.delete_many({"season": events_config.year, "game_id": game_id})
        raise

    inserted_games += 1
    inserted_rows += len(result.inserted_ids)
    print(f"game_id={game_id}: inserted {len(result.inserted_ids):,} rows")

client.close()

{"processed_games": inserted_games, "inserted_rows": inserted_rows}
